In [6]:
import json
import faiss
import numpy as np
import requests
import re

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder
)

from rank_bm25 import BM25Okapi


# =========================================================
# LOAD DOCUMENTS
# =========================================================

with open("mne_docs_test.json", "r") as f:
    documents = json.load(f)

print(f"Loaded {len(documents)} documents.")


# =========================================================
# LOAD EMBEDDINGS + FAISS INDEX
# =========================================================

embeddings = np.load("mne_embeddings.npy")

index = faiss.read_index(
    "mne_faiss.index"
)

print("Embeddings shape:", embeddings.shape)
print("FAISS index loaded.")


# =========================================================
# LOAD MODELS
# =========================================================

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print("Models loaded.")


# =========================================================
# BUILD BM25 INDEX
# =========================================================

import re

bm25_corpus = []

for doc in documents:

    text = (
        doc["function_name"]
        + " "
        + doc["description"]
    )

    tokens = re.findall(
        r"[A-Za-z0-9_.]+",
        text.lower()
    )

    bm25_corpus.append(tokens)

bm25 = BM25Okapi(bm25_corpus)

print("BM25 index created.")


# =========================================================
# QUERY
# =========================================================

target_api = "mne.filter.notch_filter"

query = target_api


# =========================================================
# QUERY EMBEDDING
# =========================================================

query_embedding = embedding_model.encode(
    [query]
)

query_embedding = np.array(
    query_embedding,
    dtype="float32"
)

print("Query embedding shape:", query_embedding.shape)


# =========================================================
# DENSE RETRIEVAL (FAISS)
# =========================================================

k_dense = 2

distances, dense_indices = index.search(
    query_embedding,
    k_dense
)

dense_results = dense_indices[0].tolist()

print("\nDense Retrieval Results:")
print(dense_results)


# =========================================================
# SPARSE RETRIEVAL (BM25)
# =========================================================

tokenized_query = query.lower().split()

bm25_scores = bm25.get_scores(
    tokenized_query
)

bm25_indices = np.argsort(
    bm25_scores
)[::-1][:1]

bm25_results = bm25_indices.tolist()

print("\nBM25 Retrieval Results:")
print(bm25_results)


# =========================================================
# HYBRID MERGE
# =========================================================

hybrid_indices = list(
    set(
        dense_results + bm25_results
    )
)

print("\nHybrid Retrieval Results:")
print(hybrid_indices)


# =========================================================
# COLLECT HYBRID DOCS
# =========================================================

hybrid_docs = []

for idx in hybrid_indices:
    hybrid_docs.append(documents[idx])

print(f"\nCollected {len(hybrid_docs)} documents.")


# =========================================================
# FUNCTION NAME FILTERING
# =========================================================

filtered_docs = []

for doc in hybrid_docs:

    if target_api in doc["function_name"]:
        filtered_docs.append(doc)

print(f"\nFiltered docs count: {len(filtered_docs)}")


# =========================================================
# PREPARE RERANKER PAIRS
# =========================================================

pairs = []

for doc in filtered_docs:

    combined_text = f"""
    Function:
    {doc['function_name']}

    Parameters:
    {json.dumps(doc['parameters'], indent=2)}
    """

    pairs.append(
        (query, combined_text)
    )

print(f"\nPrepared {len(pairs)} reranking pairs.")


# =========================================================
# CROSS-ENCODER RERANKING
# =========================================================

scores = reranker.predict(pairs)

reranked_results = list(
    zip(scores, filtered_docs)
)

reranked_results = sorted(
    reranked_results,
    key=lambda x: x[0],
    reverse=True
)


# =========================================================
# DISPLAY FINAL TOP RESULT
# =========================================================

top_doc = reranked_results[0]

score, doc = top_doc

print("\n" + "=" * 60)

print(f"Top Score: {score:.4f}")

print("\nFunction:")
print(doc["function_name"])


# =========================================================
# BUILD FINAL CONTEXT
# =========================================================

final_context = f"""

Function:
{doc["function_name"]}

Parameters:
{json.dumps(doc["parameters"], indent=2)}

"""


# =========================================================
# PROMPT CONSTRUCTION
# =========================================================

prompt = f"""
You are an API constraint and test generation system.

Generate constraints ONLY for:
{target_api}

Using ONLY the retrieved API documentation below,
generate parameter-level constraints and corresponding test cases.

For each inferred constraint provide:

1. Parameter Name
2. Constraint
3. Short Reasoning
4. Valid Example
5. Invalid pytest-style Test Case

Focus on:
- datatype constraints
- invalid input conditions
- mutually conflicting parameters
- filesystem-related failures
- boundary conditions

Rules:
- ONLY use behaviors explicitly supported by the documentation
- If a behavior is not explicitly specified,
  say: "Not explicitly specified"
- Do NOT invent undocumented parameters
- Do NOT assume hidden implementation details
- Keep outputs concise and structured

Retrieved Documentation:
{final_context}
"""


# =========================================================
# OLLAMA GENERATION
# =========================================================

url = "http://localhost:11434/api/generate"

payload = {
    "model": "qwen3:8b-q4_K_M",
    "prompt": prompt,
    "stream": False
}

response = requests.post(
    url,
    json=payload
)

result = response.json()

print("\n" + "=" * 60)
print("GENERATED CONSTRAINTS")
print("=" * 60)

print(result["response"])

Loaded 60 documents.
Embeddings shape: (60, 384)
FAISS index loaded.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Models loaded.
BM25 index created.
Query embedding shape: (1, 384)

Dense Retrieval Results:
[7, 6]

BM25 Retrieval Results:
[7]

Hybrid Retrieval Results:
[6, 7]

Collected 2 documents.

Filtered docs count: 1

Prepared 1 reranking pairs.

Top Score: 5.5661

Function:
mne.filter.notch_filter

GENERATED CONSTRAINTS
- Constraint: x array must be a valid array-like structure (e.g., numpy array).  
  - Valid example: `np.array([1, 2, 3])`  
  - Invalid test case: `x=123` (non-array)  

- Constraint: Fs must be a positive float.  
  - Valid example: `Fs=100.0`  
  - Invalid test case: `Fs=-100.0`  

- Constraint: freqs must be a float, array of floats, or None if method is 'spectrum_fit'.  
  - Valid example: `freqs=[10, 20]`  
  - Invalid test case: `freqs=None` when method is not 'spectrum_fit'  

- Constraint: filter_length must be a string (e.g., '10s'), integer, or 'auto'.  
  - Valid example: `filter_length='10s'`  
  - Invalid test case: `filter_length='10m'`  

- Constraint: notch_